# Testing Segmentation Models Before Implementations

## ResNet101 as backbone

In [1]:
import sys
sys.path.append('../')

from utils.models.resnet101 import resnet101_backbone
import torch

resnet101 = resnet101_backbone()
print(resnet101)

ResNetBackbone(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), str

In [2]:
resnet101.eval()
x = torch.randn(1, 3, 512, 512)
out = resnet101(x)
for k, v in out.items():
    print(k, v.shape)

c2 torch.Size([1, 256, 128, 128])
c3 torch.Size([1, 512, 64, 64])
c4 torch.Size([1, 1024, 32, 32])
c5 torch.Size([1, 2048, 16, 16])


## Mask R-CNN

In [2]:
from utils.models.maskRCNN import MaskRCNN

mask_rcnn = MaskRCNN(resnet101, num_classes=1, segmentation_mode=True)
print(mask_rcnn)

MaskRCNN(
  (backbone): ResNetBackbone(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(


In [3]:
mask_rcnn.eval()
img = torch.randn(1,3,512,512)
outputs = mask_rcnn(img)
print("Segmentation mask output shape:", outputs.shape)

Segmentation mask output shape: torch.Size([1, 1, 512, 512])


## MobileNetv2 as backbone

In [2]:
import sys
sys.path.append('../')
from utils.models.mobileNetv2 import MobileNetV2Backbone

mobileNetv2 = MobileNetV2Backbone(output_stride=16)
print(mobileNetv2)

MobileNetV2Backbone(
  (features): Sequential(
    (0): ConvBNReLU(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNReLU(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): ReLU6(inplace=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNReLU(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(i

In [4]:
import torch

mobileNetv2.eval()
inp = torch.randn(1, 3, 512, 512)
out = mobileNetv2(inp)
print("Input:", inp.shape)
for k, v in out.items():
    print(f"{k}: {v.shape}")

Input: torch.Size([1, 3, 512, 512])
low_level: torch.Size([1, 24, 128, 128])
high_level: torch.Size([1, 1280, 16, 16])


## DeepLabv3+

In [7]:
from utils.models.deeplabv3p import DeepLabV3Plus

device = torch.device("cpu")
deeplabv3p = DeepLabV3Plus(num_classes=1, output_stride=16, backbone_width_mult=1.0).to(device)
print(deeplabv3p)

DeepLabV3Plus(
  (backbone): MobileNetV2Backbone(
    (features): Sequential(
      (0): ConvBNReLU(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): ConvBNReLU(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): ReLU6(inplace=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): ConvBNReLU(
            (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(96, eps=1e-05, mo

In [ ]:
deeplabv3p.eval()
inp = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    out = deeplabv3p(inp)  # (1, num_classes, 512, 512)
print("Input:", inp.shape)
print("Segmentation logits:", out.shape)

Input: torch.Size([1, 3, 512, 512])
Segmentation logits: torch.Size([1, 1, 512, 512])


## U-NET

In [5]:
import sys
sys.path.append('../')
from utils.models.uNet import UNet
import torch

channels = [32, 64, 128, 256, 512]
model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True)
model.eval()
inp = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    out = model(inp)
print("Input:", inp.shape)
print("Output logits:", out.shape)

Input: torch.Size([1, 3, 512, 512])
Output logits: torch.Size([1, 1, 512, 512])
